# Lesson 06 Walkthrough: Database Design Basics

**Course:** Database Applications Development  
**Lesson:** 06 - Database Design and Normalization

---

## Learning Objectives

Today we will learn:
1. What makes a good database design
2. How to identify primary keys
3. How tables connect with foreign keys
4. Why we avoid storing duplicate data

**Keep it simple - these are the building blocks of good database design!**

---

## Setup

In [ ]:
import pandas as pd
import sqlite3

# Connect to database
conn = sqlite3.connect('nba_5seasons.db')
print("✅ Connected to NBA database")

---

## Part 1: Our Database Tables

Let's see what tables we're working with.

### Step 1: Show all tables

**What you need to do:**
- SELECT the `name` column
- FROM the `sqlite_master` table
- WHERE the `type` equals 'table'

In [ ]:
# Show all tables in the database
query = """
SELECT ________ 
FROM _______________
WHERE ________ = '________';
"""

tables = pd.read_sql(query, conn)
print("Tables in our NBA database:")
print(tables)

**We have 4 tables:**
- `teams` - information about NBA teams
- `players` - information about players
- `team_game_stats` - how teams performed in games
- `player_season_stats` - player stats for each season

---

## Part 2: Understanding Primary Keys

**Primary Key = The unique identifier for each row**

Let's look at the `teams` table:

### Step 2: View table structure

**What you need to do:**
- Use `PRAGMA table_info(teams)` to see the table structure

In [ ]:
# Look at teams table structure
query = """
PRAGMA table_info(________);
"""

schema = pd.read_sql(query, conn)
print("Teams table structure:")
print(schema)

**Notice:** The `pk` column shows `1` for `team_id` - that's our primary key!

Now let's see the actual data:

### Step 3: View sample team data

**What you need to do:**
- SELECT team_id, full_name, abbreviation, and city
- FROM the teams table
- Show only the first 5 rows

In [ ]:
# Show sample teams
query = """
SELECT ________, __________, ____________, ________
FROM ________
LIMIT ____;
"""

result = pd.read_sql(query, conn)
print("Sample teams:")
print(result)

**Key Point:** Each `team_id` appears only ONCE. That's what makes it a good primary key!

Let's verify this:

### Step 4: Verify primary key uniqueness

**What you need to do:**
- Count all rows with `COUNT(*)`
- Count unique team_ids with `COUNT(DISTINCT team_id)`
- These numbers should match!

In [ ]:
# Count total teams vs unique team_ids
query = """
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT ________) AS unique_team_ids
FROM teams;
"""

result = pd.read_sql(query, conn)
print(result)
print("\n✅ If these numbers match, every team_id is unique!")

---

## Part 3: Understanding Foreign Keys

**Foreign Key = A column that points to a primary key in another table**

Let's look at how `team_game_stats` connects to `teams`:

### Step 5: View game stats with team_id

**What you need to do:**
- SELECT season, game_id, team_id, pts, reb, and ast
- FROM team_game_stats
- Show first 5 rows

In [ ]:
# Show team_game_stats structure (simplified)
query = """
SELECT ________, ________, ________, _____, _____, _____
FROM __________________
LIMIT ____;
"""

result = pd.read_sql(query, conn)
print("Team game stats:")
print(result)

**See the `team_id` column?** That's a foreign key that points to the teams table!

Let's use it to get team names:

### Step 6: JOIN tables to get team names

**What you need to do:**
- SELECT full_name (from teams table) as 'team'
- SELECT pts and reb (from team_game_stats)
- FROM team_game_stats (alias: tgs)
- JOIN teams (alias: t) ON matching team_ids
- WHERE season equals '2021-22'
- Show first 5 rows

In [ ]:
# Join tables to get team names
query = """
SELECT 
    t.__________ AS team,          -- team full name
    tgs._____ AS points,            -- points from game stats
    tgs._____ AS rebounds           -- rebounds from game stats
FROM __________________ tgs         -- game stats table
JOIN ________ t                     -- teams table
    ON tgs.________ = t.________    -- match the team_ids
WHERE tgs.season = '________'       -- filter to 2021-22 season
LIMIT ____;
"""

result = pd.read_sql(query, conn)
print("Games with team names:")
print(result)

**This is the power of foreign keys!** We can connect tables to get more information.

---

## Part 4: Why Avoid Duplicate Data?

Let's see how much storage we save with good design.

**Bad Design:** Store team name in every game  
**Good Design:** Store team name once in teams table

### Step 7: Count team names in good design

**What you need to do:**
- Count how many rows are in the teams table
- This is how many team names we store in GOOD design

In [ ]:
# Good design: How many team names do we store?
query = """
SELECT COUNT(*) AS team_names_stored
FROM ________;
"""

good_design = pd.read_sql(query, conn)['team_names_stored'][0]
print(f"Good design: {good_design} team names stored")

### Step 8: Count how many times we'd repeat names in bad design

**What you need to do:**
- Count all games in team_game_stats
- WHERE season equals '2021-22'
- This shows how many times we'd store team names if we repeated them

In [ ]:
# Bad design: How many times would we repeat team names?
query = """
SELECT COUNT(*) AS team_names_if_repeated
FROM __________________
WHERE season = '________';
"""

bad_design = pd.read_sql(query, conn)['team_names_if_repeated'][0]
print(f"Bad design: {bad_design} team names stored")
print(f"\n💾 We save {bad_design - good_design} duplicates by using good design!")

**Also think about updates:** If a team changes its name:
- Good design: Update 1 row in teams table ✅
- Bad design: Update thousands of game records ❌

---

## Part 5: Quick Check - What Did We Learn?

Let's practice identifying good vs bad design:

In [ ]:
# Example of BAD design (for demonstration only)
query = """
CREATE TEMP TABLE bad_design AS
SELECT 
    tgs.season,
    tgs.game_id,
    t.full_name AS team_name,
    t.city AS team_city,
    tgs.pts
FROM team_game_stats tgs
JOIN teams t ON tgs.team_id = t.team_id
WHERE tgs.season = '2021-22'
LIMIT 5;
"""

conn.execute(query)

result = pd.read_sql("SELECT * FROM bad_design", conn)
print("❌ BAD Design - team info repeated in every game:")
print(result)

**Problems with this design:**
1. Team name and city stored thousands of times (waste of space)
2. If team changes name, we'd have to update thousands of rows
3. Risk of typos making data inconsistent

**Our actual database uses good design:**
- Team info stored ONCE in teams table
- Games reference teams using team_id
- Easy to update, no duplicates!

---

## Summary: Keys to Good Database Design

**1. Primary Keys**
- Every table needs a unique identifier
- Use ID numbers (like team_id, player_id)
- Never use names as primary keys

**2. Foreign Keys**
- Connect tables together
- Point to primary keys in other tables
- Enable JOINs to combine data

**3. Avoid Duplicates**
- Store each piece of info in ONE place
- Reference it with foreign keys
- Saves space and makes updates easy

**That's it! These simple rules make databases work well!**

In [ ]:
# Close connection
conn.close()
print("✅ Walkthrough complete!")